# Setting

## Library

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Matplotlib global settings
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [3]:
# ML libraries
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

In [4]:
# Helper functions & model import
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *
from model import *

## Function

In [5]:
def select_uids_by_class(df, sample_number, class_col='Class', uid_col='uid', random_state=42):
    np.random.seed(random_state)
    uids_by_class = {}
    for c in df[class_col].unique():
        uids = df[df[class_col]==c][uid_col].unique()
        n_select = min(sample_number, len(uids))
        selected_uids = np.random.choice(uids, n_select, replace=False)
        uids_by_class[c] = set(selected_uids)
    return uids_by_class

In [6]:
def filter_by_selected_uids(df, uids_by_class, class_col='Class', uid_col='uid'):
    mask = np.zeros(len(df), dtype=bool)
    for c, uids in uids_by_class.items():
        mask |= ((df[class_col]==c) & (df[uid_col].isin(uids)))
    return df[mask]

In [7]:
from sklearn.model_selection import GroupShuffleSplit

def train_test_split_by_uid(df, test_size=0.2, class_col='Class', uid_col='uid', random_state=42):
    groups = df[uid_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(df, df[class_col], groups))
    return df.iloc[train_idx], df.iloc[test_idx]

## Initial Setup

In [8]:
logtxt = ""

In [9]:
# Set experiment configs
test_name = "LightGBM_Class_Selection"
random_state = 42
test_size = 0.2
device_type = "cpu" # or gpu
n_jobs = -1
path_save = os.path.join(MODEL, test_name)
os.makedirs(path_save, exist_ok=True)

logtxt += "\nSet experiment configs\n"
logtxt += f"test_name: {test_name}\n"
logtxt += f"random_state: {random_state}\n"
logtxt += f"test_size: {test_size}\n"
logtxt += f"device_type: {device_type}\n"
logtxt += f"n_jobs: {n_jobs}\n"
logtxt += f"path_save: {path_save}\n"
logtxt += "\n"


In [10]:
path_data = os.path.join(FEATURE_BALANCED_DATA, 'features_40.csv')

logtxt += f"\nBalanced Data Set\n"

# Data

In [11]:
columns_to_use = list(data_dtype_dict.keys())

- Source to Consider

In [12]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	"LBV", 
	"TDE", 
	"Nova", 
	"M dwarf", 
	"CV",
	"SLSN",
]
logtxt += f"\nSources to consider: {sources_to_consider}\n"

In [13]:

data = pd.read_csv(
    path_data,
    engine='c', 
    # usecols=columns_to_use,
    # dtype=data_dtype_dict,
)

data['uid'] = data['uid'].astype(str)
data['Class'] = data['Class'].astype(str)

uids = data['uid'].values
classes = data['Class'].values

print(f"Balanced Data: {len(data)}")

logtxt += f"Balanced Data: {len(data)}\n"

indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data = data.iloc[indx_type_to_consider[0]]

logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n"
logtxt += "\n"


Balanced Data: 282753
10 sources to consider: 282753


- Training and Test Data

In [14]:
# - Split features/target
data1 = data.copy()

X1 = data1.drop(columns=['Sample_ID', 'Class', 'uid'])
y1 = data1['Class']
X1.fillna(-99, inplace=True)

# - Split into train/test using GroupShuffleSplit by uid
# gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
# train_idx, test_idx = next(gss.split(X, y, groups=data['uid']))
# X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
# y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# - Label encode class for ML
# label_encoder = LabelEncoder()
# y_encoded = label_encoder.fit_transform(y)
# y_train_encoded = label_encoder.fit_transform(y_train)
# y_test_encoded = label_encoder.transform(y_test)
# class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
# print("Balanced: Class mapping:", class_names)

# Tets\\sts
# classifier_type = 'normal_class_classifier'
# model_param_config = model_config[classifier_type][device_type]

In [15]:
# del data

- Define GV, but no sampling

In [16]:
# Define GV class group
gv_classes = ["Nova", "M dwarf", "CV", "LBV"]
unified_label = "GV"

# Step 1: Replace Class labels
data['Class'] = data['Class'].replace(gv_classes, unified_label)

# Step 2: Redefine sources_to_consider
sources_to_consider = [
    "AGN", 
    "Ia", 
    "II", 
    "Ibc", 
    "TDE", 
    "SLSN", 
    unified_label,  # add merged label
]
logtxt += f"\nSources to consider (with GV): {sources_to_consider}\n"

# Step 3: Filter data again with new sources
indx_type_to_consider = np.where(
    np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data2 = data.iloc[indx_type_to_consider[0]]

logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n"
logtxt += "\n"

# - Split features/target
X2 = data2.drop(columns=['Sample_ID', 'Class', 'uid'])
y2 = data2['Class']
X2.fillna(-99, inplace=True)

# # - Split into train/test using GroupShuffleSplit by uid
# gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
# train_idx, test_idx = next(gss.split(X, y, groups=data['uid']))
# X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
# y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# # - Label encode class for ML
# label_encoder = LabelEncoder()
# y_encoded = label_encoder.fit_transform(y)
# y_train_encoded = label_encoder.fit_transform(y_train)
# y_test_encoded = label_encoder.transform(y_test)
# class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
# print("Balanced: Class mapping:", class_names)

7 sources to consider: 282753


- Define GV, 1/4 sampling

In [ ]:
# Step 0: Define GV group and label
gv_classes = ["Nova", "M dwarf", "CV", "LBV"]
unified_label = "GV"

# Step 1: Initialize list to collect subsampled data
gv_subsample_list = []

# Step 2: For each GV class, sample 1/4 of unique uids
for cls in gv_classes:
    cls_data = data[data['Class'] == cls]
    unique_uids = cls_data['uid'].unique()
    n_sample = len(unique_uids) // 4

    sampled_uids = np.random.choice(unique_uids, size=n_sample, replace=False)
    sampled_data = cls_data[cls_data['uid'].isin(sampled_uids)]

    # Replace label to unified GV label
    sampled_data = sampled_data.copy()
    sampled_data['Class'] = unified_label

    gv_subsample_list.append(sampled_data)

# Step 3: Concatenate subsampled GV data
gv_data = pd.concat(gv_subsample_list, ignore_index=True)

# Step 4: Keep other classes unchanged
non_gv_data = data[~data['Class'].isin(gv_classes)]

# Step 5: Merge into final data
data = pd.concat([non_gv_data, gv_data], ignore_index=True)

# Step 6: Redefine sources_to_consider
sources_to_consider = [
    "AGN", 
    "Ia", 
    "II", 
    "Ibc", 
    "TDE", 
    "SLSN", 
    unified_label,
]
logtxt += f"\nSources to consider (with GV sampled): {sources_to_consider}\n"

# Step 7: Filter data again
indx_type_to_consider = np.where(
    np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)
print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data3 = data.iloc[indx_type_to_consider[0]]

logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n\n"

# Step 8: Train/test split & encoding
X3 = data3.drop(columns=['Sample_ID', 'Class', 'uid'])
y3 = data3['Class']
X3.fillna(-99, inplace=True)

# gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
# train_idx, test_idx = next(gss.split(X, y, groups=data['uid']))
# X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
# y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# label_encoder = LabelEncoder()
# y_encoded = label_encoder.fit_transform(y)
# y_train_encoded = label_encoder.fit_transform(y_train)
# y_test_encoded = label_encoder.transform(y_test)

# class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
# print("Balanced: Class mapping:", class_names)

- Define GV, 1/4 random sampling, but wihtin the same uid

In [ ]:
# Step 0: Define GV group and label
gv_classes = ["Nova", "M dwarf", "CV", "LBV"]
unified_label = "GV"

# Step 1: Initialize list to collect subsampled data
gv_subsample_list = []

# Step 2: For each GV class, sample a subset of rows per uid
for cls in gv_classes:
    cls_data = data[data['Class'] == cls]
    # Group by uid and sample {frac}% of each uid's samples (at least 1)
    sampled_data = cls_data.groupby('uid').apply(
        lambda df: df.sample(frac=0.25, random_state=42) if len(df) > 1 else df
    ).reset_index(drop=True)

    # Replace label to unified GV label
    sampled_data = sampled_data.copy()
    sampled_data['Class'] = unified_label
    gv_subsample_list.append(sampled_data)

# Step 3: Concatenate subsampled GV data
gv_data = pd.concat(gv_subsample_list, ignore_index=True)

# Step 4: Keep other classes unchanged
non_gv_data = data[~data['Class'].isin(gv_classes)]

# Step 5: Merge into final data
data = pd.concat([non_gv_data, gv_data], ignore_index=True)

# Step 6: Redefine sources_to_consider
sources_to_consider = [
    "AGN", 
    "Ia", 
    "II", 
    "Ibc", 
    "TDE", 
    "SLSN", 
    unified_label,
]
logtxt += f"\nSources to consider (with GV sampled): {sources_to_consider}\n"

# Step 7: Filter data again
indx_type_to_consider = np.where(
    np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)
print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data4 = data.iloc[indx_type_to_consider[0]]
logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n\n"

# Step 8: Train/test split & encoding
X4 = data4.drop(columns=['Sample_ID', 'Class', 'uid'])
y4 = data4['Class']
X4.fillna(-99, inplace=True)

# gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
# train_idx, test_idx = next(gss.split(X, y, groups=data['uid']))
# X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
# y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# label_encoder = LabelEncoder()
# y_encoded = label_encoder.fit_transform(y)
# y_train_encoded = label_encoder.fit_transform(y_train)
# y_test_encoded = label_encoder.transform(y_test)

# class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
# print("Balanced: Class mapping:", class_names)

In [ ]:
# Step 0: Define GV group and label
gv_classes = ["Nova", "M dwarf", "CV", "LBV"]
unified_label = "GV"

# Step 1: Initialize list to collect subsampled data
gv_subsample_list = []

# Step 2: For each GV class, sample a subset of rows per uid
for cls in gv_classes:
    cls_data = data[data['Class'] == cls]
    # Group by uid and sample {frac}% of each uid's samples (at least 1)
    sampled_data = cls_data.groupby('uid').apply(
        lambda df: df.sample(frac=0.4, random_state=42) if len(df) > 1 else df
    ).reset_index(drop=True)

    # Replace label to unified GV label
    sampled_data = sampled_data.copy()
    sampled_data['Class'] = unified_label
    gv_subsample_list.append(sampled_data)

# Step 3: Concatenate subsampled GV data
gv_data = pd.concat(gv_subsample_list, ignore_index=True)

# Step 4: Keep other classes unchanged
non_gv_data = data[~data['Class'].isin(gv_classes)]

# Step 5: Merge into final data
data = pd.concat([non_gv_data, gv_data], ignore_index=True)

# Step 6: Redefine sources_to_consider
sources_to_consider = [
    "AGN", 
    "Ia", 
    "II", 
    "Ibc", 
    "TDE", 
    "SLSN", 
    unified_label,
]
logtxt += f"\nSources to consider (with GV sampled): {sources_to_consider}\n"

# Step 7: Filter data again
indx_type_to_consider = np.where(
    np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)
print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data5 = data.iloc[indx_type_to_consider[0]]
logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n\n"

# Step 8: Train/test split & encoding
X5 = data5.drop(columns=['Sample_ID', 'Class', 'uid'])
y5 = data5['Class']
X5.fillna(-99, inplace=True)

# gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
# train_idx, test_idx = next(gss.split(X, y, groups=data['uid']))
# X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
# y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# label_encoder = LabelEncoder()
# y_encoded = label_encoder.fit_transform(y)
# y_train_encoded = label_encoder.fit_transform(y_train)
# y_test_encoded = label_encoder.transform(y_test)

# class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
# print("Balanced: Class mapping:", class_names)

In [ ]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	"LBV", 
	"TDE", 
	# "Nova", 
	"M dwarf", 
	"CV",
	"SLSN",
]
indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data6 = data.iloc[indx_type_to_consider[0]]

# - Split features/target
X6 = data6.drop(columns=['Sample_ID', 'Class', 'uid'])
y6 = data6['Class']
X6.fillna(-99, inplace=True)


In [ ]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	"LBV", 
	"TDE", 
	# "Nova", 
	# "M dwarf", 
	"CV",
	"SLSN",
]
indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data7 = data.iloc[indx_type_to_consider[0]]

# - Split features/target
X7 = data7.drop(columns=['Sample_ID', 'Class', 'uid'])
y7 = data7['Class']
X7.fillna(-99, inplace=True)


In [ ]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	# "LBV", 
	"TDE", 
	# "Nova", 
	# "M dwarf", 
	# "CV",
	"SLSN",
]
indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data8 = data8.iloc[indx_type_to_consider[0]]

# - Split features/target
X8 = data8.drop(columns=['Sample_ID', 'Class', 'uid'])
y8 = data8['Class']
X8.fillna(-99, inplace=True)


In [ ]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	# "LBV", 
	"TDE", 
	# "Nova", 
	# "M dwarf", 
	# "CV",
	# "SLSN",
]
indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data9 = data.iloc[indx_type_to_consider[0]]

# - Split features/target
X9 = data9.drop(columns=['Sample_ID', 'Class', 'uid'])
y9 = data9['Class']
X9.fillna(-99, inplace=True)


In [ ]:
sources_to_consider = [
	# "AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	# "LBV", 
	"TDE", 
	# "Nova", 
	# "M dwarf", 
	# "CV",
	# "SLSN",
]
indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data10 = data.iloc[indx_type_to_consider[0]]

# - Split features/target
X10 = data10.drop(columns=['Sample_ID', 'Class', 'uid'])
y10 = data10['Class']
X10.fillna(-99, inplace=True)


# Start

In [ ]:
def plot_confusion_matrix():
    import seaborn as sns
    cm = confusion_matrix(y_test, y_pred)
    labels = label_encoder.classes_
    cm_percent = (cm / cm.sum(axis=1, keepdims=True)) * 100
    n_rows, n_cols = cm.shape
    combined_matrix = np.empty_like(cm, dtype=object)
    for i in range(n_rows):
        for j in range(n_cols):
            n = cm[i, j]
            p = cm_percent[i, j]
            combined_matrix[i, j] = f"{n:,}\n({p:.1f}%)"
    plt.figure(figsize=(12, 9))
    ax = sns.heatmap(cm_percent, annot=False, fmt="", cmap="Blues",
                        xticklabels=labels, yticklabels=labels,
                        cbar_kws={'label': '[%]'}, vmin=0, vmax=100)
    plt.xlabel("Predicted Label", fontsize=16)
    plt.ylabel("True Label", fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    # plt.title(__class__.__name__.replace("Experiment", ""))
    plt.tight_layout()
    thresh = 50
    for i in range(n_rows):
        for j in range(n_cols):
            value = cm_percent[i, j]
            text_color = "white" if value > thresh else "black"
            ax.text(j+0.5, i+0.5, combined_matrix[i, j], ha='center', va='center', color=text_color, fontsize=14)

In [ ]:
params = {
    'boosting_type': 'gbdt',
    # 'objective': 'binary',               # 'multiclass' if multiclass
    'learning_rate': 0.1,
    'n_estimators': 100,
    'max_depth': -1,                     # no limit
    'num_leaves': 31,
    'min_child_samples': 20,
    'min_child_weight': 0.001,
    'subsample': 1.0,
    'subsample_freq': 0,
    'colsample_bytree': 1.0,
    'reg_alpha': 0.0,
    'reg_lambda': 0.0,
    'random_state': None,
    'n_jobs': n_jobs,
    'importance_type': 'split',
    # 'metric': 'binary_logloss',          # only used in train(), not in fit()
    'verbosity': -1,
    'device': 'cpu',                     # GPU 사용 시: 'gpu'
}

In [ ]:
from lightgbm import LGBMClassifier  # 예시로 LGBM 사용

# model = LGBMClassifier(**params)
# model.fit(
#     X_train, y_train,
#     )
# y_pred = model.predict(X_test)


In [ ]:
# plot_confusion_matrix()

In [ ]:
# report_dict = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
# acc_value = report_dict.pop("accuracy")
# report_df = pd.DataFrame(report_dict).T
# report_df

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import os
import numpy as np
import pandas as pd

summary_dict = dict()

experiment_list = [
    # (exp_name, data, X, y)
    ("all", data1, X1, y1),
    ("all_GV", data2, X2, y2),
    ("all_GV_uid_1/4", data3, X3, y3),
    ("all_GV_frac=0.25", data4, X4, y4),
    ("all_GV_frac=0.4", data5, X5, y5),
    ("no_Nova", data6, X6, y6),
    ("no_Nova_Mdwarf", data7, X7, y7),
    ("no_GV", data8, X8, y8),
    ("no_GV,SLSN", data9, X9, y9),
    ("no_GV,SLSN,AGN", data10, X10, y10),
]

for exp_name, data, X, y in experiment_list:
    print(f"\n=== 실험: {exp_name} ===")
    
    # class_names 자동 추출 (알파벳 순서)
    class_names = np.unique(y)
    
    # 데이터 split
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(X, y, groups=data['uid']))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # 모델 학습
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # 평가 리포트
    report_dict = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
    acc_value = report_dict.pop("accuracy", None)
    report_df = pd.DataFrame(report_dict).T
    report_df.loc["accuracy"] = [None, None, acc_value, report_df["support"].sum()]
    
    # confusion matrix도 저장
    cm = confusion_matrix(y_test, y_pred, labels=class_names)
    
    # summary_dict에 저장
    summary_dict[exp_name] = {
        "report_df": report_df,
        "accuracy": acc_value,
        "cm": cm,
        "class_names": class_names
    }
    
    # 파일로 저장 (옵션)
    save_dir = os.path.join(path_save, exp_name)
    os.makedirs(save_dir, exist_ok=True)
    report_df.to_csv(os.path.join(save_dir, "classification_report.csv"))
    np.savetxt(os.path.join(save_dir, "confusion_matrix.csv"), cm, delimiter=",")

In [ ]:
# 1. 실험별 accuracy, macro f1, weighted f1 비교 bar plot
exp_names = list(summary_dict.keys())
accuracy = []
macro_f1 = []
weighted_f1 = []

for exp in exp_names:
    report_df = summary_dict[exp]['report_df']
    accuracy.append(summary_dict[exp]['accuracy'])
    macro_f1.append(report_df.loc["macro avg", "f1-score"])
    weighted_f1.append(report_df.loc["weighted avg", "f1-score"])

x = np.arange(len(exp_names))
plt.figure(figsize=(12,5))
plt.bar(x-0.2, macro_f1, width=0.2, label='Macro F1')
plt.bar(x, weighted_f1, width=0.2, label='Weighted F1')
plt.bar(x+0.2, accuracy, width=0.2, label='Accuracy')
plt.xticks(x, exp_names, rotation=15)
plt.ylim(0, 1)
plt.ylabel('Score')
plt.title('Overall Performance Comparison by Experimental Setting')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(path_save, 'compare_experiments_performance.png'))
plt.close()